# 08 — Feature Engineering Workflows for Tabular Kaggle (Lesson)

## Learning Objectives
- Understand **why** each mathematical term appears and how it changes optimization/generalization.
- Apply the technique to both classification and regression.
- Compare synthetic and real-data behavior with reproducible experiments.

## Driving Question
> How do we design leakage-safe, high-impact feature pipelines for tabular competitions?

## Notebook Roadmap
Math (LaTeX) -> Synthetic Data -> Real Data (MNIST/California Housing) -> Visualizations -> Best Practices -> Exercises

### Mathematical Foundation (First Principles)

Feature engineering can be viewed as a map $\phi(x)$ from raw inputs to a richer representation:

$$
\hat{y} = f_\theta(\phi(x))
$$

Bias-variance view:

$$
\mathbb{E}[(y-\hat{y})^2] = \text{Bias}^2 + \text{Variance} + \sigma^2
$$

Good feature design reduces bias without exploding variance or leakage.

For target encoding of category $c$ with smoothing:

$$
\tilde{\mu}_c = \frac{n_c\mu_c + \alpha\mu_{\text{global}}}{n_c + \alpha}
$$

computed out-of-fold to avoid target leakage.

### Deep Equation Lineage (Term-by-Term)

We now connect every lesson to the same first-principles optimization pipeline.

1. **Population objective** (what we really care about):
$$
\mathcal{R}(\theta)=\mathbb{E}_{(x,y)\sim\mathcal{D}}[\ell(f_\theta(x),y)]
$$
2. **Empirical approximation** (what we can compute):
$$
\hat{\mathcal{R}}(\theta)=\frac{1}{N}\sum_{i=1}^{N}\ell(f_\theta(x_i),y_i)
$$
3. **Mini-batch stochastic estimate** (what we optimize per step):
$$
g_t=\frac{1}{m}\sum_{i \in \mathcal{B}_t}\nabla_\theta \ell(f_\theta(x_i),y_i)
$$
4. **Chain-rule expansion** (how gradients flow through layers):
$$
\nabla_\theta \ell
=
\frac{\partial \ell}{\partial \hat{y}}
\cdot
\frac{\partial \hat{y}}{\partial h_L}
\cdot
\frac{\partial h_L}{\partial h_{L-1}}
\cdots
\frac{\partial h_1}{\partial \theta}
$$
5. **Regularized update** (how each lesson perturbs the step):
$$
\theta_{t+1}
=
\theta_t
-\eta\left(g_t+\nabla_\theta\Omega(\theta_t)\right)
$$

| Term | Meaning | Where it appears in code |
|---|---|---|
| $\eta$ | step size controlling update magnitude | optimizer learning rate |
| $g_t$ | stochastic gradient estimate | `loss.backward()` |
| $\Omega(\theta)$ | explicit/implicit regularization | weight decay, dropout, early stopping |
| $m$ | mini-batch size | `DataLoader(..., batch_size=...)` |
| $\hat{y}=f_\theta(x)$ | model predictions | `logits = model(xb)` |

**Interpretation checkpoint:** if training is unstable, inspect whether the issue comes from gradient scale ($g_t$), step size ($\eta$), or noisy estimation (small $m$).

### Before the Code: Purpose + Mechanics

This setup cell builds a reproducible tabular-competition environment with optional XGBoost/LightGBM support.
Background: robust benchmarking requires stable folds, consistent preprocessing, and graceful fallbacks when some libraries are unavailable.

In [ ]:
import copy
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.datasets import fetch_california_housing, make_classification
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, train_test_split, cross_validate
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import roc_auc_score, mean_squared_error, roc_curve
from sklearn.ensemble import HistGradientBoostingRegressor, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

# Optional libraries: if unavailable, notebook still runs with sklearn fallbacks.
try:
    from xgboost import XGBRegressor, XGBClassifier
    XGB_AVAILABLE = True
except Exception:
    XGB_AVAILABLE = False

try:
    from lightgbm import LGBMRegressor, LGBMClassifier
    LGBM_AVAILABLE = True
except Exception:
    LGBM_AVAILABLE = False

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)


def make_ohe():
    # Compatibility helper for older/newer sklearn versions.
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=False)


def build_tabular_frame():
    # Use California Housing and add competition-style categorical/missingness structure.
    raw = fetch_california_housing(as_frame=True)
    X = raw.frame.drop(columns=['MedHouseVal']).copy()
    y = raw.frame['MedHouseVal'].astype(float).copy()

    X['income_bucket'] = pd.qcut(X['MedInc'], q=5, labels=False, duplicates='drop').astype(str)
    X['house_age_bucket'] = pd.cut(X['HouseAge'], bins=[0, 15, 30, 45, 60], include_lowest=True).astype(str)
    X['geo_bucket'] = (X['Latitude'].round(1).astype(str) + '_' + X['Longitude'].round(1).astype(str))

    # Inject mild missingness to mimic real competition cleanup requirements.
    missing_mask = np.random.rand(len(X)) < 0.03
    X.loc[missing_mask, 'AveRooms'] = np.nan
    X.loc[missing_mask, 'AveBedrms'] = np.nan
    return X, y


class TabularMLP(nn.Module):
    def __init__(self, input_dim, hidden_dims=(256, 128), dropout_p=0.15, output_dim=1):
        super().__init__()
        layers = []
        prev = input_dim
        for hidden in hidden_dims:
            layers.append(nn.Linear(prev, hidden))
            layers.append(nn.BatchNorm1d(hidden))
            layers.append(nn.ReLU())
            if dropout_p > 0:
                layers.append(nn.Dropout(dropout_p))
            prev = hidden
        layers.append(nn.Linear(prev, output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def make_tabular_preprocessor(num_cols, cat_cols):
    return ColumnTransformer(
        transformers=[
            ('num', Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())
            ]), num_cols),
            ('cat', Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('ohe', make_ohe())
            ]), cat_cols),
        ]
    )


def fit_preprocessor_matrices(preprocessor, X_train, X_valid):
    X_train_m = preprocessor.fit_transform(X_train)
    X_valid_m = preprocessor.transform(X_valid)
    if hasattr(X_train_m, 'toarray'):
        X_train_m = X_train_m.toarray()
    if hasattr(X_valid_m, 'toarray'):
        X_valid_m = X_valid_m.toarray()
    return np.asarray(X_train_m, dtype=np.float32), np.asarray(X_valid_m, dtype=np.float32)


def make_tabular_loaders(X_train_np, y_train, X_valid_np, y_valid, batch_size=256, task='regression'):
    y_train_t = torch.tensor(np.asarray(y_train, dtype=np.float32).reshape(-1, 1), dtype=torch.float32)
    y_valid_t = torch.tensor(np.asarray(y_valid, dtype=np.float32).reshape(-1, 1), dtype=torch.float32)
    train_ds = TensorDataset(torch.tensor(X_train_np, dtype=torch.float32), y_train_t)
    valid_ds = TensorDataset(torch.tensor(X_valid_np, dtype=torch.float32), y_valid_t)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    valid_loader = DataLoader(valid_ds, batch_size=batch_size, shuffle=False)
    return train_loader, valid_loader


def train_tabular_mlp(
    X_train,
    y_train,
    X_valid,
    y_valid,
    task='regression',
    hidden_dims=(256, 128),
    dropout_p=0.15,
    lr=8e-4,
    weight_decay=1e-5,
    batch_size=256,
    epochs=40,
    patience=6,
    verbose=False,
):
    num_cols = X_train.select_dtypes(include=['number']).columns.tolist()
    cat_cols = [c for c in X_train.columns if c not in num_cols]
    preprocessor = make_tabular_preprocessor(num_cols, cat_cols)
    X_train_np, X_valid_np = fit_preprocessor_matrices(preprocessor, X_train, X_valid)
    train_loader, valid_loader = make_tabular_loaders(
        X_train_np, y_train, X_valid_np, y_valid, batch_size=batch_size, task=task
    )

    model = TabularMLP(
        input_dim=X_train_np.shape[1],
        hidden_dims=hidden_dims,
        dropout_p=dropout_p,
        output_dim=1,
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.BCEWithLogitsLoss() if task == 'classification' else nn.MSELoss()

    history = {'train_loss': [], 'val_loss': []}
    best_state = copy.deepcopy(model.state_dict())
    best_val = float('inf')
    best_epoch = 1
    wait = 0
    stopped_epoch = epochs

    for epoch in range(epochs):
        model.train()
        train_losses = []
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            out = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        val_losses = []
        with torch.no_grad():
            for xb, yb in valid_loader:
                xb, yb = xb.to(device), yb.to(device)
                out = model(xb)
                val_losses.append(criterion(out, yb).item())

        train_loss = float(np.mean(train_losses))
        val_loss = float(np.mean(val_losses))
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)

        if verbose:
            print(f'Epoch {epoch+1:02d}: train={train_loss:.5f} val={val_loss:.5f}')

        if val_loss < best_val - 1e-6:
            best_val = val_loss
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch + 1
            wait = 0
        else:
            wait += 1

        if wait >= patience:
            stopped_epoch = epoch + 1
            break

    model.load_state_dict(best_state)

    with torch.no_grad():
        valid_tensor = torch.tensor(X_valid_np, dtype=torch.float32).to(device)
        valid_logits = model(valid_tensor).squeeze(1).cpu().numpy()

    if task == 'classification':
        valid_pred = 1.0 / (1.0 + np.exp(-valid_logits))
    else:
        valid_pred = valid_logits

    return {
        'model': model,
        'preprocessor': preprocessor,
        'history': history,
        'val_pred': valid_pred,
        'best_epoch': best_epoch,
        'stopped_epoch': stopped_epoch,
    }


def predict_tabular_mlp(bundle, X_df, task='regression'):
    X_np = bundle['preprocessor'].transform(X_df)
    if hasattr(X_np, 'toarray'):
        X_np = X_np.toarray()
    X_tensor = torch.tensor(np.asarray(X_np, dtype=np.float32), dtype=torch.float32).to(device)
    with torch.no_grad():
        logits = bundle['model'](X_tensor).squeeze(1).cpu().numpy()
    if task == 'classification':
        return 1.0 / (1.0 + np.exp(-logits))
    return logits


def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


TABULAR_MLP_BASE_CFG = {
    'hidden_dims': (256, 128),
    'dropout_p': 0.15,
    'lr': 8e-4,
    'weight_decay': 1e-5,
    'batch_size': 256,
    'epochs': 40,
    'patience': 6,
}


def run_tabular_mlp_cv(X_df, y, cfg, n_splits=5):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    oof_pred = np.zeros(len(X_df), dtype=np.float32)
    fold_scores = []

    for fold_idx, (tr_idx, va_idx) in enumerate(kf.split(X_df), start=1):
        X_tr, X_va = X_df.iloc[tr_idx], X_df.iloc[va_idx]
        y_tr, y_va = np.asarray(y)[tr_idx], np.asarray(y)[va_idx]
        bundle = train_tabular_mlp(X_tr, y_tr, X_va, y_va, task='regression', **cfg)
        pred = bundle['val_pred']
        score = rmse(y_va, pred)
        fold_scores.append(score)
        oof_pred[va_idx] = pred
        print(f'Fold {fold_idx}: RMSE={score:.5f}, best_epoch={bundle["best_epoch"]}, stop_epoch={bundle["stopped_epoch"]}')

    return {
        'fold_scores': fold_scores,
        'oof_pred': oof_pred,
        'rmse_mean': float(np.mean(fold_scores)),
        'rmse_std': float(np.std(fold_scores)),
    }

### After the Code: Background + Why It Can Help

This setup emphasizes reproducibility and fairness: same folds, same preprocessing skeleton, and explicit fallback behavior.
For lessons 08 and 09 we now share one PyTorch MLP pipeline (preprocessing -> loaders -> training loop -> inference) so modeling choices stay aligned.

## Real-World Feature Engineering Workflow
This notebook section mirrors practical Kaggle workflow: profile -> baseline -> engineer -> ablate -> interpret.

### Before the Code: Purpose + Mechanics

We begin with a strict baseline and a reusable feature-audit report.
Background: without baseline and audit artifacts, it is impossible to know whether engineered features truly add signal.

In [ ]:
X_raw, y = build_tabular_frame()


def audit_frame(df):
    report = pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'missing_pct': (df.isna().mean() * 100).round(2),
        'n_unique': df.nunique()
    }).sort_values(['missing_pct', 'n_unique'], ascending=[False, False])
    return report


audit = audit_frame(X_raw)
display(audit.head(12))

# Baseline PyTorch MLP on raw features using CV.
baseline_cfg = {**TABULAR_MLP_BASE_CFG, 'hidden_dims': (192, 96), 'dropout_p': 0.10, 'epochs': 32, 'patience': 5}
cv = KFold(n_splits=5, shuffle=True, random_state=SEED)
baseline_cv = run_tabular_mlp_cv(X_raw, y, baseline_cfg, n_splits=5)
print('Baseline CV RMSE mean/std:', baseline_cv['rmse_mean'], baseline_cv['rmse_std'])

top_missing = audit.sort_values('missing_pct', ascending=False).head(10).reset_index().rename(columns={'index': 'feature'})
num_cols = X_raw.select_dtypes(include=['number']).columns.tolist()
corr_numeric = X_raw[num_cols].corr()
top_corr_cols = corr_numeric.abs().mean().sort_values(ascending=False).head(8).index

fig, axes = plt.subplots(1, 3, figsize=(17, 4))
sns.barplot(data=top_missing, x='missing_pct', y='feature', ax=axes[0], color='tab:red')
axes[0].set_title('Top missingness percentages')
axes[0].set_xlabel('Missing %')

sns.histplot(y, bins=35, kde=True, ax=axes[1], color='tab:blue')
axes[1].set_title('Target distribution diagnostics')
axes[1].set_xlabel('MedHouseVal')

sns.heatmap(corr_numeric.loc[top_corr_cols, top_corr_cols], cmap='coolwarm', center=0, ax=axes[2])
axes[2].set_title('Numeric correlation heatmap (top columns)')
plt.tight_layout()
plt.show()

### After the Code: Background + Why It Can Help

This gives a reproducible anchor score and a concrete data-quality snapshot.
Any new feature block should justify itself by measurable lift and stable fold behavior.

### Before the Code: Purpose + Mechanics

Now we implement a leakage-aware engineering block including interactions, ratios, geospatial grouping, and OOF target encoding.
Background: OOF statistics simulate production-time availability and avoid target leakage inflation.

In [ ]:
def add_features(df):
    out = df.copy()

    # Ratio and interaction features grounded in domain intuition.
    out['rooms_per_household'] = out['AveRooms'] / (out['AveOccup'] + 1e-6)
    out['bedrooms_per_room'] = out['AveBedrms'] / (out['AveRooms'] + 1e-6)
    out['income_x_occupancy'] = out['MedInc'] * out['AveOccup']
    out['log_population'] = np.log1p(out['Population'].clip(lower=0))

    # Coarser geospatial grouping can expose regional nonlinear effects for MLPs once encoded.
    out['lat_lon_grid'] = out['Latitude'].round(1).astype(str) + '_' + out['Longitude'].round(1).astype(str)
    return out


def oof_target_encode(series, target, n_splits=5, smooth=20):
    series = series.astype(str)
    target = np.asarray(target)
    global_mean = target.mean()
    oof_encoded = np.zeros(len(series))
    folds = KFold(n_splits=n_splits, shuffle=True, random_state=SEED)

    for tr_idx, va_idx in folds.split(series):
        tr_s, va_s = series.iloc[tr_idx], series.iloc[va_idx]
        tr_y = target[tr_idx]
        stats = pd.DataFrame({'cat': tr_s, 'target': tr_y}).groupby('cat')['target'].agg(['mean', 'count'])
        smooth_mean = (stats['mean'] * stats['count'] + smooth * global_mean) / (stats['count'] + smooth)
        oof_encoded[va_idx] = va_s.map(smooth_mean).fillna(global_mean).values
    return oof_encoded


X_eng = add_features(X_raw)
X_eng['geo_target_oof'] = oof_target_encode(X_eng['geo_bucket'], y, n_splits=5, smooth=30)

engineered_cfg = {**TABULAR_MLP_BASE_CFG, 'hidden_dims': (256, 128), 'dropout_p': 0.18, 'epochs': 38, 'patience': 6}
engineered_cv = run_tabular_mlp_cv(X_eng, y, engineered_cfg, n_splits=5)

summary = pd.DataFrame([
    {'pipeline': 'baseline_mlp', 'rmse_mean': baseline_cv['rmse_mean'], 'rmse_std': baseline_cv['rmse_std']},
    {'pipeline': 'engineered_mlp', 'rmse_mean': engineered_cv['rmse_mean'], 'rmse_std': engineered_cv['rmse_std']},
])
summary['delta_vs_baseline'] = summary['rmse_mean'] - summary.loc[summary['pipeline'] == 'baseline_mlp', 'rmse_mean'].iloc[0]
display(summary)

fold_compare = pd.DataFrame({
    'fold': np.arange(1, len(baseline_cv['fold_scores']) + 1),
    'baseline_rmse': baseline_cv['fold_scores'],
    'engineered_rmse': engineered_cv['fold_scores']
})
fold_long = fold_compare.melt(id_vars='fold', var_name='pipeline', value_name='rmse')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.lineplot(data=fold_long, x='fold', y='rmse', hue='pipeline', marker='o', ax=axes[0])
axes[0].set_title('Fold-by-fold RMSE diagnostics (PyTorch MLP)')

sns.barplot(data=summary, x='pipeline', y='rmse_mean', ax=axes[1], palette='Set2')
axes[1].errorbar(
    x=np.arange(len(summary)),
    y=summary['rmse_mean'],
    yerr=summary['rmse_std'],
    fmt='none',
    ecolor='black',
    capsize=4
)
axes[1].set_title('Mean RMSE with fold variance')
plt.tight_layout()
plt.show()

viz_cols = ['MedInc', 'rooms_per_household', 'income_x_occupancy', 'geo_target_oof']
viz_df = X_eng[viz_cols].copy()
viz_df['target'] = y.values
pair = sns.pairplot(viz_df.sample(min(1500, len(viz_df)), random_state=SEED), corner=True, diag_kind='hist')
pair.fig.suptitle('Feature-target relationship scan', y=1.02)
plt.show()

# Fit one holdout model for residual diagnostics and feature-target sanity checks.
X_fit, X_hold, y_fit, y_hold = train_test_split(X_eng, y, test_size=0.2, random_state=SEED)
eng_bundle = train_tabular_mlp(X_fit, y_fit, X_hold, y_hold, task='regression', **engineered_cfg)
hold_pred = eng_bundle['val_pred']
hold_resid = y_hold.values - hold_pred

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].scatter(hold_pred, hold_resid, alpha=0.35, s=12)
axes[0].axhline(0.0, color='black', linestyle='--')
axes[0].set_title('Holdout residuals vs prediction (engineered MLP)')
axes[0].set_xlabel('Prediction')
axes[0].set_ylabel('Residual')

numeric_corr = X_eng.select_dtypes(include=['number']).copy()
numeric_corr['target'] = y.values
corr_rank = numeric_corr.corr(numeric_only=True)['target'].drop('target').abs().sort_values(ascending=False).head(12)
axes[1].barh(corr_rank.index[::-1], corr_rank.values[::-1], color='tab:purple')
axes[1].set_title('Top numeric signal features (|corr with target|)')
plt.tight_layout()
plt.show()

### After the Code: Background + Why It Can Help

A robust feature workflow ends with both quantitative lift and interpretability evidence.
If engineered lift is unstable across folds, simplify and re-test rather than shipping brittle complexity.

## Best Practices
- Always compare engineered pipelines against a leakage-safe baseline.
- Use out-of-fold encodings for target-based categorical statistics.
- Track each engineered block with ablations to keep only high-signal features.

## Common Pitfalls
- Applying a technique without a baseline comparison.
- Drawing conclusions from training loss only.
- Ignoring seed sensitivity and run-to-run variance.

## Explanatory Depth Checkpoints
- **Why this workflow?** Because leaderboard gains are fragile without leakage-safe validation and reproducible logs.
- **Key idea:** Strong experiments isolate one hypothesis at a time so score movement has a clear causal explanation.
- **Important pitfall:** A public-score spike can hide private overfitting; always compare fold variance and shake risk.
- **In practice:** Keep assumptions explicit, test alternatives quickly, and record rollback plans before each submission.
- **Question:** Which diagnostic would make you reject a seemingly strong model?
- **Question:** How would you defend your final submission choice to a teammate under deadline pressure?

## Exercises (Competition-Oriented)
1. **Exercise:** Re-run all experiments with a second seed and quantify score/rank variance.
2. **Exercise:** Replace one preprocessing block and document exact before/after effects.
3. **Exercise:** Perform a 3-row ablation table: baseline, +feature/model change, +tuning change.
4. **Exercise:** Add one failure analysis plot and explain what action it suggests.
5. **Exercise:** Build a strict 30-minute local runtime pipeline and maximize validation score.
6. **Question:** Which feature block improved mean score but harmed fold stability, and why?
7. **Exercise:** Reproduce your best run with one different fold schema and compare ranking movement.
8. **Exercise:** Add a simple blend and report whether shake risk improved versus single-model baseline.
9. **Question:** If compute is cut by 50%, which experiments stay and which are dropped?
10. **Exercise:** Write a short decision memo defending one final submission and one rollback candidate.